# CFP Ranker '26

## Data Collection

### Team Collection

In [1]:
%pip install requests
%pip install pandas
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# NOTE: This block of code extracts the list of FBS teams from ESPN's API and saves it to a CSV file.
# This is arguably more useful because it includes team IDs and conference information
import requests
import pandas as pd

url = "https://site.web.api.espn.com/apis/v2/sports/football/college-football/standings?group=80"
# 80 is the ID for the FBS (Football Bowl Subdivision) in ESPN's API

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json"
}

response = requests.get(url, headers=headers)
response.raise_for_status()

data = response.json()

def parse_standings_node(node, parent_conference="", parent_conf_id=""):
    """Recursively walks standings nodes to extract teams, conference names, and conference IDs."""
    teams = []
    
    current_name = node.get('name', '')
    current_id = str(node.get('id', ''))
    
    # Prevent top-level league/subdivision labels from overriding actual conference names/IDs
    is_root_label = current_name.lower() in ['fbs', 'ncaa - football bowl subdivision', 'football bowl subdivision']
    
    if is_root_label:
        conf_label = ""
        conf_id = ""
    else:
        conf_label = parent_conference if parent_conference else current_name
        conf_id = parent_conf_id if parent_conf_id else current_id

    # Extract teams if standings entries exist at this level
    entries = node.get('standings', {}).get('entries', [])
    for entry in entries:
        team = entry.get('team', {})
        if team:
            teams.append({
                "team_id": team.get('id'),
                "display_name": team.get('displayName'),
                "short_name": team.get('shortDisplayName'),
                "conference": conf_label,
                "conference_id": conf_id
            })

    # Recurse into sub-children (e.g. East/West divisions)
    for child in node.get('children', []):
        teams.extend(parse_standings_node(child, parent_conference=conf_label, parent_conf_id=conf_id))

    return teams

fbs_teams = parse_standings_node(data)

df = pd.DataFrame(fbs_teams)
df = df.drop_duplicates(subset=['team_id']).reset_index(drop=True)

df.to_csv("espn_fbs_teams.csv", index=False)
print(f"Successfully extracted {len(df)} FBS teams with conference IDs.")

Successfully extracted 138 FBS teams with conference IDs.


### Schedule Parsing

In [ ]:
import json
import requests
import pandas as pd
import time
from datetime import datetime

# --- CONFIG ---
YEAR = 2026  # Works for any year (e.g., 2023, 2024, 2025, 2026)
FBS_TEAMS_FILE = "../data/espn_fbs_teams.csv"

# --- ENDPOINTS & HEADERS ---
SCOREBOARD_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"
SUMMARY_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event="

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json"
}

params = {
    "dates": "20261017",
    "groups": "80",
    "limit": "300"
}

response = requests.get(SCOREBOARD_URL, headers=HEADERS, params=params)
target_game = None

try:
	games = response.json().get("events", [])
	if response.status_code == 200:
		for event in games:
			game_name = event.get("name", "")
			if "Alabama" in game_name and "Tennessee" in game_name:
				target_game = event
				break
except Exception as e:
	print(f"Error fetching games: {e}")
	games = []

if target_game:
	print(f"Target game found: {target_game.get('name', '')}")
else:
	print("Target game not found.")

game_id = target_game.get("id", "") if target_game else None
if game_id:
	print(f"Game ID: {game_id}")
else:
	print("Game ID not found.")

def export_boxscore_to_csv(game_id, output_filename="boxscore.csv"):
    # Base URL for the summary endpoint
    summary_url = f"https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event={game_id}"
    
    headers = HEADERS

    print(f"Fetching summary data for Game ID: {game_id}...")
    response = requests.get(summary_url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        
        # 2. Navigate to the Team Boxscore section
        teams_data = data.get("boxscore", {}).get("teams", [])
        
        if not teams_data:
            print("No box score data available yet.")
            return

        # 3. Parse the data into a list of dictionaries
        rows = []
        for team in teams_data:
            team_name = team.get("team", {}).get("displayName", "Unknown")
            stats = team.get("statistics", [])
            
            # Create a dictionary for the row, starting with the team name
            team_row = {"Team": team_name}
            
            # Loop through each stat (e.g., 'totalYards', 'firstDowns') and add it to the row
            for stat in stats:
                stat_name = stat.get("label", stat.get("name"))
                stat_value = stat.get("displayValue")
                team_row[stat_name] = stat_value
                
            rows.append(team_row)

        # 4. Use Pandas to convert the rows to a DataFrame and export to CSV
        df = pd.DataFrame(rows)
        
        # Optional: Transpose the DataFrame if you want the stats as rows instead of columns
        df = df.set_index("Team").T
        df.reset_index(inplace=True)
        df.rename(columns={"index": "Stat Category"}, inplace=True)
        
        df.to_csv(output_filename, index=False)
        print(f"\nSuccess! Data exported to {output_filename}")
        print(df.head())
        
    else:
        print(f"Failed to retrieve data. Status Code: {response.status_code}")

# Use the game ID we found in the previous step
TARGET_GAME_ID = game_id # Replace with your dynamic Game ID
export_boxscore_to_csv(TARGET_GAME_ID, f"Tennessee_Alabama_Boxscore_{TARGET_GAME_ID}.csv")

import requests
import json

def save_full_game_json(game_id, output_filename="full_game_data.json"):
    summary_url = f"https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event={game_id}"
    
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json"
    }

    print(f"Fetching full summary JSON for Game ID: {game_id}...")
    response = requests.get(summary_url, headers=headers)

    if response.status_code == 200:
        # Parse the JSON response
        data = response.json()
        
        # Open a new file in write mode and dump the JSON into it
        # indent=4 makes the file highly readable (pretty-printed)
        with open(output_filename, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=4)
            
        print(f"Success! The full JSON payload has been saved to '{output_filename}'.")
    else:
        print(f"Failed to retrieve data. Status Code: {response.status_code}")

# The Game ID for your specific matchup
TARGET_GAME_ID = game_id

save_full_game_json(TARGET_GAME_ID, "Tennessee_Alabama_Full_Summary.json")

Target game found: Alabama Crimson Tide at Tennessee Volunteers
Game ID: 401856723
Fetching summary data for Game ID: 401856723...

Success! Data exported to Tennessee_Alabama_Boxscore_401856723.csv
Team        Stat Category Alabama Crimson Tide Tennessee Volunteers
0               1st Downs                   12                   20
1     3rd down efficiency                 2-13                 6-15
2     4th down efficiency                  1-3                  1-1
3             Total Yards                  201                  347
4                 Passing                  147                  197
Fetching full summary JSON for Game ID: 401856723...
Success! The full JSON payload has been saved to 'Tennessee_Alabama_Full_Summary.json'.


In [ ]:
def explore_json_structure(filename="Tennessee_Alabama_Full_Summary.json"):
    # 1. Load the JSON file
    with open(filename, "r", encoding="utf-8") as file:
        data = json.load(file)

    # 2. Identify all top-level keys (The main "folders" of the JSON)
    print("--- TOP-LEVEL JSON SECTIONS ---")
    for key in data.keys():
        print(f"- {key}")
    
    print("\n")

    # 3. Explore a specific tabular section (e.g., Play-by-Play data)
    # The 'plays' node is a list of dictionaries, perfect for a spreadsheet
    if "plays" in data:
        print("--- PLAY-BY-PLAY COLUMNS ---")
        plays_data = data["plays"]
        
        # Convert the list of plays into a pandas DataFrame
        plays_df = pd.DataFrame(plays_data)
        
        # Print all the available columns we can extract
        for col in plays_df.columns:
            print(f"- {col}")
            
        print(f"\nTotal plays found: {len(plays_df)}")
        
        # Optional: Save just the play-by-play data to its own CSV!
        # plays_df.to_csv("Play_by_Play.csv", index=False)
    else:
        print("No play-by-play data found in this JSON.")

explore_json_structure("Tennessee_Alabama_Full_Summary.json")